# Wav2Lip — GPU Colab POC

Runs our ONNX Wav2Lip pipeline on Colab. Produces the **plain** and **GPEN-enhanced** outputs, then reports **timing** and **quality metrics** so you can compare with the MuseTalk and LatentSync notebooks.

> Set **Runtime → Change runtime type → GPU** first.

## Step 0 — confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 — install deps (GPU ONNX Runtime)

Installs **`onnxruntime-gpu==1.19.2`** (matches Colab's CUDA 12) so Wav2Lip runs on the **T4** — the run cell's `providers:` line will then show `CUDAExecutionProvider`. If you hit a `libcudart.so.XX` import error (Colab's CUDA drifted), fall back to CPU (Wav2Lip is tiny, so CPU is only a few seconds): `!pip -q install onnxruntime` then **Runtime → Restart session**.

In [ ]:
!pip -q install onnxruntime-gpu==1.19.2 opencv-python-headless librosa soundfile imageio-ffmpeg huggingface_hub tqdm scipy

## Step 2 — download models (Wav2Lip + YuNet + GPEN)

In [ ]:
import urllib.request, shutil, os
from huggingface_hub import hf_hub_download
shutil.copy(hf_hub_download('wanesoft/faceswap_pack', 'wav2lip_gan.onnx'), 'wav2lip_gan.onnx')
urllib.request.urlretrieve('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', 'yunet.onnx')
urllib.request.urlretrieve('https://github.com/harisreedhar/Face-Upscalers-ONNX/releases/download/Models/GPEN-BFR-512.onnx', 'gpen_bfr_512.onnx')
print('models:', [f for f in os.listdir('.') if f.endswith('.onnx')])

## Step 3 — write the inference script

In [ ]:
%%writefile wav2lip_gpu.py
# Self-contained Wav2Lip (ONNX) inference. Auto-uses GPU if onnxruntime-gpu is present.
import argparse, os, subprocess
import cv2, numpy as np, onnxruntime as ort, librosa, imageio_ffmpeg
from scipy import signal
from tqdm import tqdm

SR, NFFT, HOP, WIN, NMELS, FMIN, FMAX = 16000, 800, 200, 800, 80, 55, 7600
MINDB, REFDB, PREEMPH, MAXABS = -100, 20, 0.97, 4.0
IMG, STEP = 96, 16
_mb = None

def melspectrogram(w):
    global _mb
    if _mb is None:
        _mb = librosa.filters.mel(sr=SR, n_fft=NFFT, n_mels=NMELS, fmin=FMIN, fmax=FMAX)
    y = signal.lfilter([1, -PREEMPH], [1], w)
    D = librosa.stft(y=y, n_fft=NFFT, hop_length=HOP, win_length=WIN)
    S = 20 * np.log10(np.maximum(1e-5, np.dot(_mb, np.abs(D)))) - REFDB
    return np.clip((2 * MAXABS) * ((S - MINDB) / (-MINDB)) - MAXABS, -MAXABS, MAXABS)

def providers():
    a = ort.get_available_providers()
    return (["CUDAExecutionProvider", "CPUExecutionProvider"]
            if "CUDAExecutionProvider" in a else ["CPUExecutionProvider"])

_fc = {}
def feather(bh, bw):
    if (bh, bw) not in _fc:
        m = np.zeros((bh, bw), np.float32); b = max(2, int(min(bh, bw) * 0.12))
        m[b:bh - b, b:bw - b] = 1.0
        m = cv2.GaussianBlur(m, (0, 0), b / 2.0, b / 2.0); _fc[(bh, bw)] = m[:, :, None]
    return _fc[(bh, bw)]

def detect(det, frame, pad=(0.0, 0.10, 0.0, 0.0)):
    h, w = frame.shape[:2]; det.setInputSize((w, h)); _, f = det.detect(frame)
    if f is None or len(f) == 0: raise RuntimeError("no face detected")
    x = max(f, key=lambda z: z[-1]); x1, y1, bw, bh = x[:4].astype(int)
    x2, y2 = x1 + bw, y1 + bh; pt, pb, pl, pr = pad
    return (max(0, int(x1 - pl * bw)), max(0, int(y1 - pt * bh)),
            min(w, int(x2 + pr * bw)), min(h, int(y2 + pb * bh)))

class GPEN:
    def __init__(self):
        so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.s = ort.InferenceSession("gpen_bfr_512.onnx", sess_options=so, providers=providers())
        self.i = self.s.get_inputs()[0].name; self.o = self.s.get_outputs()[0].name
    def enhance(self, face):
        h, w = face.shape[:2]
        f = cv2.resize(face, (512, 512)); f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        f = ((f - 0.5) / 0.5).transpose(2, 0, 1)[None].astype(np.float32)
        o = self.s.run([self.o], {self.i: f})[0][0].transpose(1, 2, 0)
        o = ((np.clip(o, -1, 1) + 1) / 2 * 255).astype(np.uint8)
        return cv2.resize(cv2.cvtColor(o, cv2.COLOR_RGB2BGR), (w, h))

def read_frames(path):
    if os.path.splitext(path)[1].lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
        return [cv2.imread(path)], True
    cap = cv2.VideoCapture(path); fr = []
    while True:
        ok, f = cap.read()
        if not ok: break
        fr.append(f)
    cap.release(); return fr, False

def mel_chunks(mel, fps):
    ch = []; mult = 80.0 / fps; i = 0; n = mel.shape[1]
    while True:
        s = int(i * mult)
        if s + STEP > n: ch.append(mel[:, n - STEP:]); break
        ch.append(mel[:, s:s + STEP]); i += 1
    return ch

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--face", required=True); ap.add_argument("--audio", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--fps", type=float, default=25.0)
    ap.add_argument("--enhance", choices=["none", "gpen"], default="none")
    ap.add_argument("--batch", type=int, default=64)
    a = ap.parse_args()
    print("providers:", providers())
    sess = ort.InferenceSession("wav2lip_gan.onnx", providers=providers())
    in_mel, in_face = [i.name for i in sess.get_inputs()]; out_name = sess.get_outputs()[0].name
    det = cv2.FaceDetectorYN.create("yunet.onnx", "", (320, 320), 0.6, 0.3, 5000)
    gpen = GPEN() if a.enhance == "gpen" else None
    mel = melspectrogram(librosa.load(a.audio, sr=SR)[0]); chunks = mel_chunks(mel, a.fps)
    frames, still = read_frames(a.face); n = len(chunks)
    if still:
        frames = [frames[0]] * n; boxes = [detect(det, frames[0])] * n
    else:
        seq = frames + frames[::-1]; frames = [seq[i % len(seq)] for i in range(n)]
        boxes = [detect(det, f) for f in frames]
    h, w = frames[0].shape[:2]; ff = imageio_ffmpeg.get_ffmpeg_exe()
    p = subprocess.Popen([ff, "-y", "-hide_banner", "-loglevel", "error", "-f", "rawvideo",
        "-pix_fmt", "bgr24", "-s", f"{w}x{h}", "-r", str(a.fps), "-i", "-", "-i", a.audio,
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest", a.out],
        stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    for s in tqdm(range(0, n, a.batch)):
        e = min(n, s + a.batch); ib, mb, meta = [], [], []
        for i in range(s, e):
            fr = frames[i].copy(); x1, y1, x2, y2 = boxes[i]
            face = cv2.resize(fr[y1:y2, x1:x2], (IMG, IMG)); mask = face.copy(); mask[IMG // 2:] = 0
            ib.append(np.concatenate([mask, face], axis=2) / 255.0); mb.append(chunks[i])
            meta.append((fr, (x1, y1, x2, y2)))
        img_np = np.asarray(ib, np.float32).transpose(0, 3, 1, 2)
        mel_np = np.asarray(mb, np.float32)[:, None, :, :]
        pred = sess.run([out_name], {in_mel: mel_np, in_face: img_np})[0].transpose(0, 2, 3, 1) * 255.0
        for j, (fr, (x1, y1, x2, y2)) in enumerate(meta):
            bw, bh = x2 - x1, y2 - y1
            pp = cv2.resize(np.clip(pred[j], 0, 255).astype(np.uint8), (bw, bh))
            m = feather(bh, bw); roi = fr[y1:y2, x1:x2].astype(np.float32)
            bl = (pp.astype(np.float32) * m + roi * (1 - m)).astype(np.uint8)
            if gpen is not None:
                bl = (gpen.enhance(bl).astype(np.float32) * m + roi * (1 - m)).astype(np.uint8)
            fr[y1:y2, x1:x2] = bl; p.stdin.write(np.ascontiguousarray(fr).tobytes())
    p.stdin.close(); err = p.stderr.read().decode("ignore")
    if p.wait() != 0: raise RuntimeError(err)
    print("done ->", a.out)

if __name__ == "__main__":
    main()


## Step 4 — inputs
Upload a **face image or video** + an **audio** file.

In [ ]:
from google.colab import files
up = files.upload()
names = list(up.keys())
FACE = [n for n in names if n.lower().endswith(('.png','.jpg','.jpeg','.mp4','.mov'))][0]
AUDIO = [n for n in names if n.lower().endswith(('.wav','.mp3','.m4a'))][0]
print('face =', FACE, '| audio =', AUDIO)

## Step 5 — run (plain) + timing

In [ ]:
import time; _t = time.time()
!python wav2lip_gpu.py --face "$FACE" --audio "$AUDIO" --out wav2lip_demo.mp4
print(f'[TIME] Wav2Lip plain end-to-end: {time.time()-_t:.1f}s')

## Step 6 — run (GPEN-enhanced) + timing

In [ ]:
import time; _t = time.time()
!python wav2lip_gpu.py --face "$FACE" --audio "$AUDIO" --out wav2lip_enhanced.mp4 --enhance gpen
print(f'[TIME] Wav2Lip + GPEN end-to-end: {time.time()-_t:.1f}s')

## Step 7 — quality metrics

In [ ]:
# Quality metrics (reference-free): CSIM identity + mouth/face sharpness.
# Same definitions as the local wav2lip/evaluate.py, so numbers are comparable
# across notebooks. Needs OpenCV YuNet + SFace (downloaded here if missing).
import urllib.request, os, cv2, numpy as np
_M = {
 "yunet.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
 "sface.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
}
for _n, _u in _M.items():
    if not os.path.exists(_n):
        urllib.request.urlretrieve(_u, _n)
_det = cv2.FaceDetectorYN.create("yunet.onnx", "", (320, 320), 0.6, 0.3, 5000)
_rec = cv2.FaceRecognizerSF.create("sface.onnx", "")

def _big(f):
    h, w = f.shape[:2]; _det.setInputSize((w, h)); _, fc = _det.detect(f)
    return None if fc is None or len(fc) == 0 else max(fc, key=lambda z: z[-1])

def _lap(g):
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def _mouth(fr, face):
    lm = face[4:14].reshape(5, 2); (rx, ry), (lx, ly) = lm[3], lm[4]
    cx, cy = int((rx + lx) / 2), int((ry + ly) / 2)
    mw = int(abs(lx - rx) * 1.6) or 40; mh = int(mw * 0.7)
    c = fr[max(0, cy - mh // 2):cy + mh // 2, max(0, cx - mw // 2):cx + mw // 2]
    return 0.0 if c.size == 0 else _lap(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY))

def _first(path):
    if path.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
        return cv2.imread(path)
    cap = cv2.VideoCapture(path); ok, fr = cap.read(); cap.release(); return fr

def metrics(video, source, every=5):
    src = _first(source); sf = _big(src)
    sfeat = _rec.feature(_rec.alignCrop(src, sf))
    cap = cv2.VideoCapture(video); cs, ms, fs = [], [], []; i = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i % every == 0:
            f = _big(fr)
            if f is not None:
                cs.append(float(_rec.match(sfeat, _rec.feature(_rec.alignCrop(fr, f)),
                                           cv2.FaceRecognizerSF_FR_COSINE)))
                ms.append(_mouth(fr, f)); x, y, bw, bh = f[:4].astype(int)
                roi = fr[max(0, y):y + bh, max(0, x):x + bw]
                if roi.size:
                    fs.append(_lap(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)))
        i += 1
    cap.release()
    _m = lambda a: float(np.mean(a)) if a else 0.0
    print(f"CSIM(identity)={_m(cs):.3f}   mouth_sharp={_m(ms):.1f}   "
          f"face_sharp={_m(fs):.1f}   frames={len(cs)}   [CSIM>0.6=same person]")


In [ ]:
print('== Wav2Lip =='); metrics('wav2lip_demo.mp4', FACE)
print('== Wav2Lip + GPEN =='); metrics('wav2lip_enhanced.mp4', FACE)

## Step 8 — preview

In [ ]:
from IPython.display import HTML
from base64 import b64encode
def show(p):
    d = b64encode(open(p,'rb').read()).decode()
    return HTML(f'<b>{p}</b><br><video width=420 controls><source src="data:video/mp4;base64,{d}" type="video/mp4"></video>')
display(show('wav2lip_demo.mp4')); display(show('wav2lip_enhanced.mp4'))